In [ ]:
# Summary

- Loading data into vector databases

In [1]:
import os

In [3]:
%pip install -qU azure-identity azure-keyvault-secrets

Note: you may need to restart the kernel to use updated packages.


# Optional - get api keys from key vault
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

In [ ]:
secret_client = SecretClient(vault_url=f"https://{os.getenv('KEYVAULT_NAME')}.vault.azure.net/", 
                            credential=DefaultAzureCredential())
for page in secret_client.list_properties_of_secrets():
    print(page.name)

ANTHROPIC-API-KEY
AZURE-SQL-PASSWORD
CHAINLIT-AUTH-SECRET
ClOUDFARE-AI-GATEWAY
GITHUB-CR-PAT
GOOGLE-API-KEY
LANGSMITH-API-KEY
MEEA-DATABASE-URL
MEEA-DATABASE-URL-ASYNC
MEEA-JWT-SECRET-KEY
my-secret
OAUTH-GITHUB-CLIENT-ID
OAUTH-GITHUB-CLIENT-SECRET
OPENAI-API-KEY
PG-VECTOR-CONNECTION
TAVILY-API-KEY


In [109]:
from IPython.display import display_markdown

## Settings

In [13]:
from pydantic_settings import BaseSettings, SettingsConfigDict

In [ ]:
class VectorStoreSettings(BaseSettings):
    model_config = SettingsConfigDict(
        frozen=True,
        case_sensitive=True,
        extra='ignore',
        yaml_file=None,
        json_file=None,
        env_file='.env',
    )

    KEYVAULT_NAME: str

    # Chroma
    CHROMA_HTTP_URL: str = "your-chroma-url"
    CHROMA_API_KEY: str = secret_client.get_secret(name="CHROMA-API-KEY").value

    # Qdrant
    QDRANT_HTTP_URL: str = "your-qdrant-url"
    QDRANT_API_KEY: str = secret_client.get_secret(name="QDRANT-API-KEY").value

    # Weaviate
    WEAVIATE_HTTP_URL: str = "your-weaviate-url"
    WEAVIATE_API_KEY: str = secret_client.get_secret(name="WEAVIATE-API-KEY").value

    # PG Vector Store
    PG_VECTOR_CONNECTION: str = secret_client.get_secret(name="PG-VECTOR-CONNECTION").value

    # Tools
    TAVILY_API_KEY: str = secret_client.get_secret(name="TAVILY-API-KEY").value

settings = VectorStoreSettings()

In [18]:
print(settings.KEYVAULT_NAME)

kvmeeaneu01


## 1. Chroma

In [11]:
import chromadb
print(chromadb.__version__)

0.5.23


In [22]:
# help(chromadb.errors.ChromaError)

> **Hint**: Since chroma is running inside an autoscaled container, youmight initially see a 502 error. This is because the container is still starting up. Please wait a few minutes and try again.

In [25]:
# chroma_client = chromadb.PersistentClient(path="./chroma")
chroma_client = chromadb.HttpClient(
    host=settings.CHROMA_HTTP_URL,
    settings=chromadb.Settings(
        # If module not found error, upgrade chromadb with `pip install -U chromadb`
        chroma_client_auth_provider="chromadb.auth.token_authn.TokenAuthClientProvider",
        chroma_client_auth_credentials=settings.CHROMA_API_KEY)
)
print(chroma_client.heartbeat())
print("Chroma HTTP client READY!\nLet's create a collection...")
collection = chroma_client.get_or_create_collection("lc_debug_collection")
collection.add(ids=["1", "2", "3"], documents=["a", "b", "c"])

print("Verify that the collection is created..")
chroma_client.list_collections()

1735513310035325424
Chroma HTTP client READY!
Let's create a collection...
Verify that the collection is created..


[Collection(name=next-js-docs),
 Collection(name=angular-curated-knowledge),
 Collection(name=lc_debug_collection),
 Collection(name=next-js-official-examples)]

## Qdrant

In [29]:
from qdrant_client import QdrantClient

In [66]:
qdrant_client = QdrantClient(        
    url=f'{settings.QDRANT_HTTP_URL}:443',
    api_key=settings.QDRANT_API_KEY,
    timeout=60,)

print(qdrant_client.get_collections())

collections=[CollectionDescription(name='Qdrant Web Documentation')]


## Weaviate

In [26]:
import weaviate
from weaviate.classes.init import Auth

In [49]:
# weaviate_client = weaviate.connect_to_custom(
#     http_host=settings.WEAVIATE_HTTP_URL,
#     http_port=443,
#     http_secure=False,
#     grpc_host=settings.WEAVIATE_HTTP_URL,  
#     grpc_port=50051,
#     grpc_secure=True,        
#     auth_credentials=Auth.api_key(str(settings.WEAVIATE_API_KEY)),
#     skip_init_checks=True

# )

# print(weaviate_client.is_ready())  # Should print: `True` if everything is working correctly
# weaviate_client.close()

## PGVector

In [52]:
import os
import traceback
from typing import Annotated

import psycopg2

from llama_index.llms.openai import OpenAI
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore

In [68]:
def init_pg_vector_store(connection_string: str = settings.PG_VECTOR_CONNECTION,
                         db_name: str = "vectordb",
                         index_name: str = "default_index",
                         n_dims: Annotated[int, "The number of dimensions in each vector"] = 1536) -> PGVectorStore:
    connection_string = connection_string or os.environ["PG_VECTOR_CONNECTION"]
    vector_store = PGVectorStore.from_params(
        connection_string=connection_string,
        database=db_name,
        table_name=index_name,
        embed_dim=n_dims,
        port=5433,   # even though provided on the conn, seems to be parsing errors probably due to sqlalchemy version mismatch
        hybrid_search=False,
        use_jsonb=True,
        cache_ok=True,
        debug=False
    )
    return vector_store


def clear_index_if_exists(index_name: str, connection_string: str = settings.PG_VECTOR_CONNECTION):
    try:
        print(f"Attemptiong to [CLEAR] index={index_name}")
        connection_string = connection_string or os.environ["PG_VECTOR_CONNECTION"]
        conn = psycopg2.connect(connection_string)
        conn.autocommit = True

        with conn.cursor() as c:
            c.execute(f"DROP  TABLE IF EXISTS {index_name}")
        print(f"[Clear] index={index_name} ok")
    except Exception as e:
        print(f"Error clearing index={index_name}: {e}")
        traceback.print_exc()


def load_directory_to_pg_vector(directory: str,
                         index_name: str = "default_index",
                         db_name: str = "vectordb",
                         overwrite: bool = False,
                         n_dims: Annotated[int, "The number of dimensions in each vector"] = 1536) -> VectorStoreIndex:
    
    documents = SimpleDirectoryReader(input_dir=directory,
                                      raise_on_error=True,
                                      exclude_hidden=True,
                                      exclude=[".bazel", ".gitignore"],
                                      recursive=True).load_data()
    
    if overwrite:
        clear_index_if_exists(index_name)
        
    vector_store = init_pg_vector_store(
        db_name=db_name,
        index_name=index_name,
        n_dims=n_dims
    )
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context, show_progress=True
    )
    return index


def get_pg_vector_index(index_name: str = "default_index",
                        n_dims: int = 1536) -> VectorStoreIndex:
    vector_store = init_pg_vector_store(
        index_name=index_name,
        n_dims=n_dims
    )
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(storage_context=storage_context)
    return index

In [63]:
os.environ["OPENAI_API_KEY"] = secret_client.get_secret(name="OPENAI-API-KEY").value

In [64]:
print("Running test query")
print('=' * 40)
index = load_directory_to_pg_vector(directory="./files", overwrite=True)
query_engine = index.as_query_engine()
response = query_engine.query("Log vs span?")
print("\nResponse:\n", response)

Running test query
Attemptiong to [CLEAR] index=default_index


/home/ndamulelo/miniconda/envs/py12/lib/python3.12/site-packages/llama_index/core/indices/base.py:110: DeprecationWarning: Call to deprecated method get_doc_id. ('get_doc_id' is deprecated, access the 'id_' property instead.) -- Deprecated since version 0.12.2.
  docstore.set_document_hash(doc.get_doc_id(), doc.hash)


[Clear] index=default_index ok


Generating embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.32it/s]



Response:
 Log and span are both essential components in understanding and monitoring the behavior of distributed systems. Logs are records of events or actions that have taken place within a system, providing detailed information about specific occurrences. On the other hand, spans are part of distributed traces and represent individual timed operations within a transaction, helping to track the flow of requests through a system. Logs offer detailed event-specific information, while spans provide a structured way to trace the path of requests across multiple services in a distributed system.


## Seed Vector Databases

In [72]:
from llama_index.readers.web import TrafilaturaWebReader
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore

In [69]:
documents = TrafilaturaWebReader().load_data(
    [
        "https://platform.openai.com/docs/guides/prompt-engineering",
        "https://cloud.google.com/discover/what-is-prompt-engineering",
        "https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview",
    ]
)

In [70]:
len(documents)

3

In [71]:
documents[-1]

Document(id_='https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/overview', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='This guide focuses on success criteria that are controllable through prompt engineering.\nNot every success criteria or failing eval is best solved by prompt engineering. For example, latency and cost can be sometimes more easily improved by selecting a different model.\nPrompt engineering is far faster than other methods of model behavior control, such as finetuning, and can often yield leaps in performance in far less time. Here are some reasons to consider prompt engineering over finetuning:\nResource efficiency: Fine-tuning requires high-end GPUs and large memory, while prompt engineering only needs text input, making it much more resource-friendly.\nCost-e

In [73]:
def load_to_chroma(documents, collection_name) -> VectorStoreIndex:
    print(f"Loading {len(documents)} documents to Chroma collection={collection_name}..")

    chroma_collection = chroma_client.create_collection(collection_name, get_or_create=True)
    vector_store = ChromaVectorStore(chroma_collection)

    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context, show_progress=True
    )
    print("Chroma Indexing done!")
    print(chroma_client.list_collections())
    return index

prompt_engineering_index = load_to_chroma(documents, "prompt_engineering")

Loading 3 documents to Chroma collection=prompt_engineering..


/home/ndamulelo/miniconda/envs/py12/lib/python3.12/site-packages/llama_index/core/indices/base.py:110: DeprecationWarning: Call to deprecated method get_doc_id. ('get_doc_id' is deprecated, access the 'id_' property instead.) -- Deprecated since version 0.12.2.
  docstore.set_document_hash(doc.get_doc_id(), doc.hash)
Parsing nodes: 100%|██████████| 3/3 [00:00<00:00, 39.38it/s]


Some nodes are missing content, skipping them...


Generating embeddings: 100%|██████████| 6/6 [00:02<00:00,  2.16it/s]


Chroma Indexing done!
[Collection(name=next-js-docs), Collection(name=angular-curated-knowledge), Collection(name=lc_debug_collection), Collection(name=prompt_engineering), Collection(name=next-js-official-examples)]


In [85]:
def load_to_pg_vector(documents, index_name) -> VectorStoreIndex:
    print(f"Loading {len(documents)} documents to PG Vector index={index_name}..")

    vector_store = init_pg_vector_store(index_name=index_name)
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context, show_progress=True
    )
    print("PG Vector Indexing done!")
    return index

llm_agents_documents = TrafilaturaWebReader().load_data(
    [
        "https://www.promptingguide.ai/research/llm-agents",
        "https://python.langchain.com/docs/tutorials/agents/",
        "https://techcommunity.microsoft.com/blog/azure-ai-services-blog/building-a-multimodal-multi-agent-framework-with-azure-openai-assistant-api/4084007",
        "https://docs.llamaindex.ai/en/stable/examples/agent/openai_agent/",
        "https://community.openai.com/t/biggest-pains-with-llm-agents-assistants-api-autogen-etc/578745/18",
    ]
)

llm_agents_introduction_index = load_to_pg_vector(llm_agents_documents, "llm_agents_introduction_index")

/home/ndamulelo/miniconda/envs/py12/lib/python3.12/site-packages/llama_index/core/indices/base.py:110: DeprecationWarning: Call to deprecated method get_doc_id. ('get_doc_id' is deprecated, access the 'id_' property instead.) -- Deprecated since version 0.12.2.
  docstore.set_document_hash(doc.get_doc_id(), doc.hash)


Loading 5 documents to PG Vector index=llm_agents_introduction_index..


Generating embeddings: 100%|██████████| 25/25 [00:03<00:00,  7.91it/s]


PG Vector Indexing done!


In [ ]:
response = generate(prompt="", output_schema=None)
decision = generate_decision(prompt="", options=["yes", "no"])

# RAG Agent(s)

In [50]:
from llama_index.agent.openai import OpenAIAgent
from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.agent import ReActAgent
from llama_index.core.llms import LLM, ChatMessage
from llama_index.core.tools import QueryEngineTool
from llama_index.embeddings.openai import OpenAIEmbedding, OpenAIEmbeddingModelType

from llama_index.llms.openai import OpenAI
from llama_index.tools.tavily_research.base import TavilyToolSpec
from llama_index.vector_stores.qdrant import QdrantVectorStore

In [81]:
embed_model = OpenAIEmbedding(embed_batch_size=10,
                              model=OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL)
llm = OpenAI(temperature=0.2, model="gpt-4o-mini", max_tokens=820)

In [78]:
def vector_store_to_query_engine(vector_store):
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_vector_store(vector_store)
    return index.as_query_engine()


vector_store_to_query_engine(
            vector_store=QdrantVectorStore(client=qdrant_client, collection_name="Qdrant Web Documentation", enable_hybrid=False)
        )

In [86]:
rag_tools = [

    QueryEngineTool.from_defaults(
        prompt_engineering_index.as_query_engine(),
        name="prompt_engineering_index",
        description="Trusted guides on prompt engineering for AI models. Useful to understand how to optimise prompts to get the best results from AI models.",
    ),

    QueryEngineTool.from_defaults(
        VectorStoreIndex.from_vector_store(
            vector_store=QdrantVectorStore(client=qdrant_client, collection_name="Qdrant Web Documentation", enable_hybrid=False)
        ),
        name="qdrant_docs_index",
        description="Qdrant documentation for the web. Useful to understand how to use Qdrant for vector storage and search. Especially for building AI applications.",
    ),


    QueryEngineTool.from_defaults(
        llm_agents_introduction_index.as_query_engine(),
        name="llm_agents_introduction_index",
        description="Introduction to LLM agents. Useful to understand how to build AI agents that can handle complex tasks with minimal supervision.",
    ),

] + TavilyToolSpec(api_key=settings.TAVILY_API_KEY).to_tool_list()

In [87]:
llm_provider = "openai"
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)  # Need to use a strong model for reasoning parts

cls = OpenAIAgent if llm_provider == "openai" else ReActAgent
rag_agent = cls.from_tools(
    rag_tools,
    llm=llm,
    memory=None,  # Use this to persist the chat history
    verbose=True,
    max_function_calls=5
)

In [ ]:
user_input = "Good evening, how can I help you today?"
chat_history = []

while True:
    response = rag_agent.stream_chat(user_input)

    response_parts = []
    print("\n\nAssistant: ", end="", flush=True)
    for r in response.response_gen:
        print(r, end="", flush=True)
        response_parts.append(r)

    chat_history.append(ChatMessage(role="assistant", content=" ".join(response_parts)))
    user_input = input("\n\nUser (Press 'q' to quit): ")
    if user_input.lower().lower() == "q":
        break

## Loading the Vector Index tools Dynamically

In [105]:
from chromadb.api.models import Collection

def get_chroma_query_engine_tools(client: chromadb.HttpClient, llm=None) -> QueryEngineTool:
    tools = []
    for collection in client.list_collections():  # type: Collection
        print(type(collection), str(collection))
        print('\t', vars(collection))
        print('\tDimensions:', collection._model.dimension)
        # Todo:
        # List collections
        # For each collection, find which embedding model was used
        # Else assume its the default model
        # Call generate() to get auto-generated description
        # Return QueryEngineTool.from_defaults

        vector_store = ChromaVectorStore.from_collection(collection)
        embed = OpenAIEmbedding(embed_batch_size=10,
                                dimensions=collection._model.dimension or 1536, # or 768,
                                model=OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL)

        tools.append(
            QueryEngineTool.from_defaults(
                name=collection.name,
                description="knowledge base containing information on " + collection.name,
                query_engine=VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed).as_query_engine(llm=llm)
            )
        )
    return tools

chroma_db_tools = get_chroma_query_engine_tools(chroma_client)
print(f"Found {len(chroma_db_tools)} Chroma collection tools")

<class 'chromadb.api.models.Collection.Collection'> Collection(name=next-js-docs)
	 {'_client': <chromadb.api.fastapi.FastAPI object at 0x7f5e177e4dd0>, '_model': Collection(id=UUID('110c4d0c-307d-4081-97f4-fc9cecbc16a7'), name='next-js-docs', configuration_json={'hnsw_configuration': {'space': 'l2', 'ef_construction': 100, 'ef_search': 10, 'num_threads': 4, 'M': 16, 'resize_factor': 1.2, 'batch_size': 100, 'sync_threshold': 1000, '_type': 'HNSWConfigurationInternal'}, '_type': 'CollectionConfigurationInternal'}, metadata=None, dimension=1536, tenant='default_tenant', database='default_database', version=0, log_position=0), '_embedding_function': <chromadb.utils.embedding_functions.onnx_mini_lm_l6_v2.ONNXMiniLM_L6_V2 object at 0x7f5e5db17b60>, '_data_loader': None}
	Dimensions: 1536
<class 'chromadb.api.models.Collection.Collection'> Collection(name=angular-curated-knowledge)
	 {'_client': <chromadb.api.fastapi.FastAPI object at 0x7f5e177e4dd0>, '_model': Collection(id=UUID('82d17088-e

In [106]:
chroma_db_tools[0].metadata

ToolMetadata(description='knowledge base containing information on next-js-docs', name='next-js-docs', fn_schema=<class 'llama_index.core.tools.types.DefaultToolFnSchema'>, return_direct=False)

In [107]:
chroma_test_agent = OpenAIAgent.from_tools(
    chroma_db_tools,
    llm=llm,
    memory=None,  # Use this to persist the chat history
    verbose=True,
    max_function_calls=5
)

In [110]:
display_markdown(chroma_test_agent.chat("What is a standalone component in Angular?"))

Added user message to memory: What is a standalone component in Angular?
=== Calling Function ===
Calling function: angular-curated-knowledge with args: {"input":"What is a standalone component in Angular?"}
Got output: A standalone component in Angular is a self-contained component that can be created and used without the need for an NgModule. It is marked with the `standalone: true` flag, allowing it to specify its dependencies directly through imports. This simplifies the authoring experience by eliminating the requirement for NgModules, making it easier to build and manage Angular applications. Standalone components can also be used alongside existing NgModule-based components, directives, and pipes, enabling incremental adoption of this new style in existing applications.



In [152]:
response = await chroma_test_agent.astream_chat("is 2024 a leap year?")
# print(type(response))
# print(vars(response))

Added user message to memory: is 2024 a leap year?


In [146]:
type(response)

llama_index.core.chat_engine.types.StreamingAgentChatResponse

In [154]:
async for r in response.async_response_gen():
    print(r)
    print(type(r))

Yes
<class 'str'>
,
<class 'str'>
 
<class 'str'>
202
<class 'str'>
4
<class 'str'>
 is
<class 'str'>
 a
<class 'str'>
 leap
<class 'str'>
 year
<class 'str'>
.
<class 'str'>
 Leap
<class 'str'>
 years
<class 'str'>
 occur
<class 'str'>
 every
<class 'str'>
 four
<class 'str'>
 years
<class 'str'>
,
<class 'str'>
 and
<class 'str'>
 since
<class 'str'>
 
<class 'str'>
202
<class 'str'>
4
<class 'str'>
 is
<class 'str'>
 divisible
<class 'str'>
 by
<class 'str'>
 
<class 'str'>
4
<class 'str'>
,
<class 'str'>
 it
<class 'str'>
 qualifies
<class 'str'>
 as
<class 'str'>
 a
<class 'str'>
 leap
<class 'str'>
 year
<class 'str'>
.
<class 'str'>
 In
<class 'str'>
 a
<class 'str'>
 leap
<class 'str'>
 year
<class 'str'>
,
<class 'str'>
 February
<class 'str'>
 has
<class 'str'>
 
<class 'str'>
29
<class 'str'>
 days
<class 'str'>
 instead
<class 'str'>
 of
<class 'str'>
 the
<class 'str'>
 usual
<class 'str'>
 
<class 'str'>
28
<class 'str'>
.
<class 'str'>


In [162]:
client = chroma_test_agent

In [159]:
response = client.stream_chat("Is 2024 a leap year?")
for chunk in response.response_gen:
    print(chunk, end="", flush=True)

Added user message to memory: Is 2024 a leap year?
Yes, 2024 is a leap year. Leap years occur every four years, and since 2024 is divisible by 4, it qualifies as a leap year. In a leap year, February has 29 days instead of the usual 28.

In [161]:
response = await client.astream_chat("Is 2024 a leap year?")
async for chunk in response.async_response_gen():
    print(chunk, end="", flush=True)

Added user message to memory: Is 2024 a leap year?
Yes, 2024 is a leap year. Leap years occur every four years, and since 2024 is divisible by 4, it qualifies as a leap year. In a leap year, February has 29 days instead of the usual 28.